In [1]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from dataclasses import dataclass
from typing import Optional
from enum import Enum
from datetime import datetime, timezone

builder = (SparkSession.builder.appName("EP3-Calidad").master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.compression.codec", "zstd"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

class Severidad(Enum):
    OK = "OK"; WARNING = "WARNING"; ERROR = "ERROR"; CRITICO = "CRITICO"

@dataclass
class ResultadoValidacion:
    nombre: str
    columna: Optional[str]
    valor: float
    threshold: float
    pasa: bool
    severidad: Severidad
    mensaje: str

resultados = []

def registrar(nombre, columna, valor, threshold, pasa, sev, msg):
    resultados.append(ResultadoValidacion(nombre, columna, valor, threshold, pasa, sev, msg))
    estado = "✓" if pasa else ("⚠" if sev == Severidad.WARNING else "✗")
    print(f"{estado} [{sev.value:7s}] {nombre}: {msg}")
    return pasa

In [2]:
df = spark.read.format("delta").load("outputs/silver_delta")
total = df.count()
print(f"Filas: {total:,}")

nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
    if n:
        print(f"{c:22s} {n:>10,}  {n/total*100:6.2f}%")

Filas: 7,728,329
flag_corregido          7,728,279  100.00%
Weather_Timestamp         120,228    1.56%
Sunrise_Sunset             23,246    0.30%
Civil_Twilight             23,246    0.30%
Nautical_Twilight          23,246    0.30%
Astronomical_Twilight      23,246    0.30%
Airport_Code               22,635    0.29%
Street                     10,869    0.14%
Timezone                    7,808    0.10%
Zipcode                     1,915    0.02%
Distance_mi                    86    0.00%


In [3]:
cols_num = ["Severity","Start_Lat","Start_Lng","Distance_mi","Duration_min",
            "Temperature_F","Humidity_pct","Pressure_in","Visibility_mi",
            "Wind_Speed_mph","Precipitation_in"]
fila = df.agg(*[x for c in cols_num for x in (F.min(c).alias(f"min_{c}"), F.max(c).alias(f"max_{c}"))]).collect()[0].asDict()
for c in cols_num:
    print(f"{c:18s} min={fila[f'min_{c}']}  max={fila[f'max_{c}']}")

Severity           min=1  max=4
Start_Lat          min=24.5548  max=49.002201
Start_Lng          min=-124.623833  max=-67.113167
Distance_mi        min=0.0  max=99.952
Duration_min       min=1.2166666666666666  max=2812999.0
Temperature_F      min=-58.0  max=129.2
Humidity_pct       min=1.0  max=100.0
Pressure_in        min=0.0  max=58.63
Visibility_mi      min=0.0  max=140.0
Wind_Speed_mph     min=0.0  max=1087.0
Precipitation_in   min=0.0  max=36.47


In [4]:
COLS_CRITICAS = ["ID", "Severity", "Start_Time", "Start_Lat", "Start_Lng"]
COLS_OPCIONALES = {"Weather_Timestamp": 5.0, "Sunrise_Sunset": 1.0,
                   "Airport_Code": 1.0, "Timezone": 1.0, "Distance_mi": 0.01}
RANGOS = {
    "Severity": (1, 4),
    "Start_Lat": (24.0, 50.0),
    "Start_Lng": (-125.0, -66.0),
    "Distance_mi": (0, 100),
    "Duration_min": (0, 10080),
    "Temperature_F": (-70, 135),
    "Humidity_pct": (0, 100),
    "Pressure_in": (20.0, 32.5),
    "Visibility_mi": (0, 100),
    "Wind_Speed_mph": (0, 250),
    "Precipitation_in": (0, 12),
}

df = spark.read.format("delta").load("outputs/silver_delta")
total = df.count()

exprs = [F.sum(F.col(c).isNull().cast("int")).alias(f"nul_{c}")
         for c in COLS_CRITICAS + list(COLS_OPCIONALES)]
exprs += [F.sum(((F.col(c) < lo) | (F.col(c) > hi)).cast("int")).alias(f"rng_{c}")
          for c, (lo, hi) in RANGOS.items()]
exprs.append(F.sum((F.col("End_Time") < F.col("Start_Time")).cast("int")).alias("tiempo_inv"))
M = df.agg(*exprs).collect()[0].asDict()
print(f"Filas: {total:,} | métricas calculadas: {len(M)}")

Filas: 7,728,329 | métricas calculadas: 22


In [5]:
for c in COLS_CRITICAS:
    n = M[f"nul_{c}"] or 0
    pct = n / total * 100
    registrar(f"nulos_{c}", c, pct, 0.0, n == 0,
              Severidad.OK if n == 0 else Severidad.CRITICO,
              f"{n:,} nulos ({pct:.4f}%), threshold 0%")

for c, thr in COLS_OPCIONALES.items():
    n = M[f"nul_{c}"] or 0
    pct = n / total * 100
    pasa = pct <= thr
    sev = Severidad.OK if pasa else (Severidad.WARNING if pct <= thr * 2 else Severidad.ERROR)
    registrar(f"nulos_{c}", c, pct, thr, pasa, sev,
              f"{n:,} nulos ({pct:.4f}%), threshold {thr}%")

✓ [OK     ] nulos_ID: 0 nulos (0.0000%), threshold 0%
✓ [OK     ] nulos_Severity: 0 nulos (0.0000%), threshold 0%
✓ [OK     ] nulos_Start_Time: 0 nulos (0.0000%), threshold 0%
✓ [OK     ] nulos_Start_Lat: 0 nulos (0.0000%), threshold 0%
✓ [OK     ] nulos_Start_Lng: 0 nulos (0.0000%), threshold 0%
✓ [OK     ] nulos_Weather_Timestamp: 120,228 nulos (1.5557%), threshold 5.0%
✓ [OK     ] nulos_Sunrise_Sunset: 23,246 nulos (0.3008%), threshold 1.0%
✓ [OK     ] nulos_Airport_Code: 22,635 nulos (0.2929%), threshold 1.0%
✓ [OK     ] nulos_Timezone: 7,808 nulos (0.1010%), threshold 1.0%
✓ [OK     ] nulos_Distance_mi: 86 nulos (0.0011%), threshold 0.01%


In [6]:
for c, (lo, hi) in RANGOS.items():
    n = M[f"rng_{c}"] or 0
    pct = n / total * 100
    sev = Severidad.OK if n == 0 else (Severidad.WARNING if pct <= 0.1 else Severidad.ERROR)
    registrar(f"rango_{c}", c, pct, 0.0, n == 0, sev,
              f"{n:,} fuera de [{lo}, {hi}] ({pct:.4f}%)")

✓ [OK     ] rango_Severity: 0 fuera de [1, 4] (0.0000%)
✓ [OK     ] rango_Start_Lat: 0 fuera de [24.0, 50.0] (0.0000%)
✓ [OK     ] rango_Start_Lng: 0 fuera de [-125.0, -66.0] (0.0000%)
✓ [OK     ] rango_Distance_mi: 0 fuera de [0, 100] (0.0000%)
⚠ [WARNING] rango_Duration_min: 6,825 fuera de [0, 10080] (0.0883%)
✓ [OK     ] rango_Temperature_F: 0 fuera de [-70, 135] (0.0000%)
✓ [OK     ] rango_Humidity_pct: 0 fuera de [0, 100] (0.0000%)
⚠ [WARNING] rango_Pressure_in: 451 fuera de [20.0, 32.5] (0.0058%)
⚠ [WARNING] rango_Visibility_mi: 15 fuera de [0, 100] (0.0002%)
⚠ [WARNING] rango_Wind_Speed_mph: 21 fuera de [0, 250] (0.0003%)
⚠ [WARNING] rango_Precipitation_in: 7 fuera de [0, 12] (0.0001%)


In [7]:
n_dup = total - df.select("ID").distinct().count()
registrar("unicidad_ID", "ID", float(n_dup), 0.0, n_dup == 0,
          Severidad.OK if n_dup == 0 else Severidad.ERROR, f"{n_dup:,} duplicados")

n_inv = M["tiempo_inv"] or 0
registrar("consistencia_End_ge_Start", "End_Time", n_inv / total * 100, 0.0, n_inv == 0,
          Severidad.OK if n_inv == 0 else Severidad.ERROR,
          f"{n_inv:,} registros con End_Time < Start_Time")

✓ [OK     ] unicidad_ID: 0 duplicados
✓ [OK     ] consistencia_End_ge_Start: 0 registros con End_Time < Start_Time


True

In [8]:
conteos = {s: 0 for s in Severidad}
for r in resultados:
    conteos[r.severidad] += 1
for s, n in conteos.items():
    print(f"{s.value:8s}: {n}")
n_err = conteos[Severidad.ERROR] + conteos[Severidad.CRITICO]
print("\nPIPELINE BLOQUEADO" if n_err else "\nPIPELINE OK")

OK      : 18
ERROR   : 0
CRITICO : 0

PIPELINE OK


In [9]:
schema_calidad = StructType([
    StructField("ts_ejecucion", TimestampType(), False),
    StructField("nombre", StringType(), False),
    StructField("columna", StringType(), True),
    StructField("valor", DoubleType(), True),
    StructField("threshold", DoubleType(), True),
    StructField("pasa", BooleanType(), False),
    StructField("severidad", StringType(), False),
    StructField("mensaje", StringType(), True),
])
ts = datetime.now(timezone.utc)
filas = [(ts, r.nombre, r.columna, float(r.valor), float(r.threshold),
          r.pasa, r.severidad.value, r.mensaje) for r in resultados]
spark.createDataFrame(filas, schema_calidad).write.format("delta") \
    .mode("append").save("outputs/calidad_reporte_delta")
print(f"Guardadas {len(filas)} validaciones")

Guardadas 23 validaciones


In [10]:
for r in resultados:
    if r.severidad != Severidad.OK:
        print(f"{r.severidad.value:8s} {r.nombre:28s} {r.mensaje}")

WARNING  rango_Duration_min           6,825 fuera de [0, 10080] (0.0883%)
WARNING  rango_Pressure_in            451 fuera de [20.0, 32.5] (0.0058%)
WARNING  rango_Visibility_mi          15 fuera de [0, 100] (0.0002%)
WARNING  rango_Wind_Speed_mph         21 fuera de [0, 250] (0.0003%)
WARNING  rango_Precipitation_in       7 fuera de [0, 12] (0.0001%)


In [11]:
from delta.tables import DeltaTable
tabla = DeltaTable.forPath(spark, "outputs/silver_delta")

cols_warn = [r.columna for r in resultados
             if r.severidad == Severidad.WARNING and r.nombre.startswith("rango_")]
print("Columnas a corregir:", cols_warn)

for c in cols_warn:
    lo, hi = RANGOS[c]
    tabla.update(condition=(F.col(c) < lo) | (F.col(c) > hi),
                 set={c: F.lit(None).cast("double")})
    print(f"UPDATE aplicado en {c}")

tabla.history().select("version", "operation").show(10)

Columnas a corregir: ['Duration_min', 'Pressure_in', 'Visibility_mi', 'Wind_Speed_mph', 'Precipitation_in']
UPDATE aplicado en Duration_min
UPDATE aplicado en Pressure_in
UPDATE aplicado en Visibility_mi
UPDATE aplicado en Wind_Speed_mph
UPDATE aplicado en Precipitation_in
+-------+------------+
|version|   operation|
+-------+------------+
|     15|      UPDATE|
|     14|      UPDATE|
|     13|      UPDATE|
|     12|      UPDATE|
|     11|      UPDATE|
|     10|  VACUUM END|
|      9|VACUUM START|
|      8|  VACUUM END|
|      7|VACUUM START|
|      6|  VACUUM END|
+-------+------------+
only showing top 10 rows



In [12]:
(spark.read.format("delta").load("outputs/calidad_reporte_delta")
    .groupBy("ts_ejecucion", "severidad").count()
    .orderBy("ts_ejecucion", "severidad").show(truncate=False))

+--------------------------+---------+-----+
|ts_ejecucion              |severidad|count|
+--------------------------+---------+-----+
|2026-11-05 21:32:18.997419|OK       |18   |
|2026-11-05 21:32:18.997419|WARNING  |5    |
+--------------------------+---------+-----+



### ¿Por qué guardar el reporte en Delta con append y no en CSV?
- Cada ejecución agrega filas con su `ts_ejecucion`, así que se conserva todo el historial sin pisar corridas anteriores.
- Delta garantiza escrituras atómicas y esquema fijo, y permite consultar con SQL y comparar corridas.
- Para detectar degradación: agrupar por `nombre` y `ts_ejecucion` y vigilar si el `valor` (porcentaje de violaciones) sube entre corridas, o si una validación pasa de OK a WARNING y de WARNING a ERROR.

In [13]:
d10 = spark.read.format("delta").option("versionAsOf", 10).load("outputs/silver_delta")
d15 = spark.read.format("delta").load("outputs/silver_delta")

def fuera(d):
    return d.agg(*[F.sum(((F.col(c) < lo) | (F.col(c) > hi)).cast("int")).alias(c)
                   for c, (lo, hi) in RANGOS.items() if c in cols_warn]).collect()[0].asDict()

print("v10 (antes):", fuera(d10))
print("v15 (después):", fuera(d15))

v10 (antes): {'Duration_min': 6825, 'Pressure_in': 451, 'Visibility_mi': 15, 'Wind_Speed_mph': 21, 'Precipitation_in': 7}
v15 (después): {'Duration_min': 0, 'Pressure_in': 0, 'Visibility_mi': 0, 'Wind_Speed_mph': 0, 'Precipitation_in': 0}
